# Notes

code for producing GOIT pipelines summary stats, and for calculating landing page stats

this is saved as an Excel file, which Baird copies/pastes into the existing summary tables information on the drive here:
https://docs.google.com/spreadsheets/d/1OYH6D7c-D0FsL5GzBGijtkmvQCTkBUclj-UVoOieUFo/edit

In [28]:
import pandas
import numpy
import pygsheets
import datetime
import re
import pytz

In [29]:
# define the excel file to save tables in
current_time = datetime.datetime.now(pytz.timezone('US/Eastern')).strftime("%Y-%m-%d_T%H%M%S")

In [30]:
fuel_type = 'Gas'
#fuel_type = 'Oil'
#fuel_type = 'NGL'

## import data

In [31]:
gc = pygsheets.authorize(service_account_env_var='GDRIVE_API_CREDENTIALS')
#spreadsheet = gc.open_by_key('1WaBMIdfRWqSqXUw7_cKXo3RipyhPdnNN8flqEYfMZIA') # file to use for gas pipelines Dec 2023
#spreadsheet = gc.open_by_key('1foPLE6K-uqFlaYgLPAUxzeXfDO5wOOqE7tibNHeqTek') # CURRENT sheet
spreadsheet = gc.open_by_key('1CktxlI1RgYUvKtL0iaNjnZszXFVjSj0JKgPnxkbm414') # Jan 2025 sheet
# spreadsheet = gc.open_by_key('1OXybaZOn0f2ONB6d_J0A3SG2bJ660C2Kr8fuc5o8cjs') # dec 2024 dataset for GGIT release

gas_pipes = spreadsheet.worksheet('title', 'Gas pipelines').get_as_df(start='A3')
oil_pipes = spreadsheet.worksheet('title', 'Oil/NGL pipelines').get_as_df(start='A3')

gas_pipes = gas_pipes.drop('WKTFormat', axis=1) # delete WKTFormat column
oil_pipes = oil_pipes.drop('WKTFormat', axis=1)
pipes_df_orig = gas_pipes.copy() #pandas.concat([oil_pipes, gas_pipes], ignore_index=True)

#get other relevant sheets
country_ratios_df = spreadsheet.worksheet('title', 'Country ratios by pipeline').get_as_df()
owners_df_orig = spreadsheet.worksheet('title', 'Pipeline operators/owners').get_as_df(start='A2')

country_ratios_df = country_ratios_df.loc[country_ratios_df.Wiki!='']

# remove empty cells for pipes, owners
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['PipelineName']!='']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig['Wiki']!='']
pipes_df_orig = pipes_df_orig.loc[pipes_df_orig.Fuel==fuel_type]

owners_df_orig = owners_df_orig.loc[owners_df_orig['ProjectID']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig['Wiki']!='']
owners_df_orig = owners_df_orig.loc[owners_df_orig.Status!='N/A']

owners_df_orig.set_index('ProjectID', inplace=True)

parent_metadata_df = spreadsheet.worksheet('title', 'Parent metadata (3/3)').get_as_df(start='A3')
parent_metadata_df.set_index('Parent', inplace=True)

In [32]:
country_ratios_df.replace('--', numpy.nan, inplace=True)

owners_df_orig.replace('',numpy.nan,inplace=True)
owners_df_orig.replace('--',numpy.nan,inplace=True)

pipes_df_orig.replace('--',numpy.nan,inplace=True)

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55122/1702877721.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  country_ratios_df.replace('--', numpy.nan, inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55122/1702877721.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  owners_df_orig.replace('--',numpy.nan,inplace=True)
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55122/1702877721.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a f

In [33]:
region_df_orig = spreadsheet.worksheet('title', 'Country dictionary').get_as_df(start='A2')

#region_name = 'Global'; region_df_touse = region_df_orig.copy()
#region_name = 'AsiaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AsiaGasTracker=='Yes']
region_name = 'EuroGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.EuroGasTracker=='Yes']
#region_name = 'AfricaGasTracker'; region_df_touse = region_df_orig.loc[region_df_orig.AfricaGasTracker=='Yes']
# region_name = 'LatinAmericaTracker'; region_df_touse = region_df_orig.loc[region_df_orig.LatinAmericaTracker=='Yes']
#region_df_agt.copy()

#region_df_touse = region_df_orig.copy()

In [34]:
region_df_touse_cleaned = region_df_touse.loc[(region_df_touse.Region!='--')&
                                            (region_df_touse.SubRegion!='--')]
multiindex_region_subregion = region_df_touse_cleaned.groupby(['Region','SubRegion'])['Country'].count().index
multiindex_region_subregion

MultiIndex([(  'Asia',    'Western Asia'),
            ('Europe',  'Eastern Europe'),
            ('Europe', 'Northern Europe'),
            ('Europe', 'Southern Europe'),
            ('Europe',  'Western Europe')],
           names=['Region', 'SubRegion'])

## file names with regional specifics

In [35]:
if fuel_type=='Gas':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-gas-pipelines-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='NGL':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-NGL-pipelines-'+str(datetime.date.today())+'.xlsx')
if fuel_type=='Oil':
    excel_writer = pandas.ExcelWriter(region_name+'-summary-sheets-Oil-pipelines-'+str(datetime.date.today())+'.xlsx')

### create country-specific dataframes for region, country_ratios_df, owners_df

In [36]:
egt_projectids = pipes_df_orig.loc[pipes_df_orig.EuropeTracker=='yes'].ProjectID.tolist()
pci6_projectids = pipes_df_orig.loc[pipes_df_orig.PCI6=='yes'].ProjectID.tolist()

In [37]:
# country_ratios_df_touse = country_ratios_df.loc[country_ratios_df['Country'].str.contains(
#                                             '|'.join(region_df_touse['Country'].tolist()))]

# owners_df_touse = owners_df_orig.loc[owners_df_orig['Countries'].str.contains(
#                                             '|'.join(region_df_touse['Country'].tolist()))]

# pipes_df_touse = pipes_df_orig.loc[pipes_df_orig['Countries'].str.contains(
#                                             '|'.join(region_df_touse['Country'].tolist()))]

country_ratios_df_touse = country_ratios_df.loc[country_ratios_df.ProjectID.isin(egt_projectids)]
pipes_df_touse = pipes_df_orig.loc[pipes_df_orig.ProjectID.isin(egt_projectids)]

In [38]:
country_ratios_df

,PipelineName,SegmentName,ProjectID,Country,LengthEstimateKmByCountry,LengthPerCountryFraction,Region,SubRegion,RegionOld,PipelineBubbleRegion,...,Parent,H2Status,H2Type,CancelledYear,ProposalYear,ConstructionYear,ShelvedYear,StartYearEarliest,StartCountry,EndCountry
0,Alberta Clipper Oil Pipeline,,P0001,Canada,1066.328784,0.680000,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2010,Canada,United States
1,Alberta Clipper Oil Pipeline,,P0001,United States,512.042577,0.320000,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2010,Canada,United States
2,Athabasca Oil Pipeline,,P0002,Canada,522.239984,1.000000,Americas,Northern America,North America,North America,...,Enbridge Inc [88.43%]; unknown [unknown %],NaN,NaN,,,,,1999,Canada,Canada
3,Bakken Expansion Pipeline,,P0004,Canada,150.276903,0.595268,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2013,United States,Canada
4,Bakken Expansion Pipeline,,P0004,United States,102.1756,0.400000,Americas,Northern America,North America,North America,...,Enbridge Inc [100.00%],NaN,NaN,,,,,2013,United States,Canada
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6645,Araucária-Cuiabá Oil Pipeline,Araucária-Jataizinho,P7195,Brazil,350.267885,1.000000,Americas,Latin America and the Caribbean,Latin America and the Caribbean,Latin America,...,NaN,NaN,NaN,,2022,,,NaN,Brazil,Brazil
6646,Araucária-Cuiabá Oil Pipeline,Jataizinho-Presidente Prudente,P7196,Brazil,140.342687,1.000000,Americas,Latin America and the Caribbean,Latin America and the Caribbean,Latin America,...,NaN,NaN,NaN,,2022,,,NaN,Brazil,Brazil
6647,Araucária-Cuiabá Oil Pipeline,Presidente Prudente-Campo Grande,P7197,Brazil,421.159047,1.000000,Americas,Latin America and the Caribbean,Latin America and the Caribbean,Latin America,...,NaN,NaN,NaN,,2022,,,NaN,Brazil,Brazil
6648,Araucária-Cuiabá Oil Pipeline,Campo Grande-Rondonópolis,P7198,Brazil,457.94597,1.000000,Americas,Latin America and the Caribbean,Latin America and the Caribbean,Latin America,...,NaN,NaN,NaN,,2022,,,NaN,Brazil,Brazil


In [39]:
pipes_df_touse.head()

,PipelineNetworkGrouping,PipelineName,SegmentName,Wiki,ProjectID,Status,Status [ref],Researcher,LastUpdated,Fuel,...,PCI6ProjectCode,PipelineDirectionality,QCCOwner,QCCOwner [ref],ImpactedByRussiaUkraineInvasion,EGTImport,ChinesePipelineType,ChineseNetworkPrimary,ChineseNetworkSecondary,ChineseClassificationNotes
5,,Reganosa Gas Pipeline,Guitiriz-As Pontes,https://www.gem.wiki/Reganosa_Gas_Pipeline,P5523,operating,,HH,2023-07-23,Gas,...,,,,,,,,,,
6,,Reganosa Gas Pipeline,As Pontes-Mugardos,https://www.gem.wiki/Reganosa_Gas_Pipeline,P5524,operating,,HH,2023-07-23,Gas,...,,,,,,,,,,
7,,Reganosa Gas Pipeline,Mugardos-Abegondo,https://www.gem.wiki/Reganosa_Gas_Pipeline,P5525,operating,,HH,2023-07-23,Gas,...,,,,,,,,,,
8,,Trans Europa Naturgas Pipeline,,https://www.gem.wiki/Trans_Europa_Naturgas_Pip...,P0723,operating,,HH,2022-07-06,Gas,...,,,,,,,,,,
10,,Adriatica Pipeline,Campochiaro-Sulmona,https://www.gem.wiki/Adriatica_Pipeline,P1344,operating,,HH,2022-07-07,Gas,...,,,,,,,,,,


### sum LengthMergedKmByCountry and MergedKmByRegion

In [40]:
status_list = ['proposed', 
               'construction', 
               'shelved', 
               'cancelled', 
               'operating', 
               'idle', 
               'mothballed', 
               'retired']
country_list = sorted(list(set(country_ratios_df_touse['Country'])))
region_list = sorted(list(set(country_ratios_df_touse['Region'])))

In [41]:
excel_status_list = ['proposed', 
                     'construction', 
                     'in development (proposed + construction)', 
                     'shelved', 
                     'cancelled', 
                     'operating', 
                     'idle', 
                     'mothballed', 
                     'retired']
excel_status_list_with_countries = ['Country']+excel_status_list

In [42]:
country_ratios_df_subset = country_ratios_df_touse.loc[country_ratios_df_touse['Fuel']==fuel_type]

km_by_country = pandas.DataFrame(columns=status_list, index=country_list)
km_by_region = pandas.DataFrame(columns=status_list, index=multiindex_region_subregion)

print('===country-level calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_country[status] = country_ratios_df_subset_status.groupby('Country')['LengthMergedKmByCountry'].sum()

print('===regional calculations===')
for status in status_list:
    print(status)
    country_ratios_df_subset_status = country_ratios_df_subset[country_ratios_df_subset['Status']==status]
    km_by_region[status] = country_ratios_df_subset_status.groupby(['Region','SubRegion'])['LengthMergedKmByCountry'].sum()

# fille NaN with 0.0
km_by_region = km_by_region.fillna(0)
km_by_country = km_by_country.fillna(0)

km_by_region['in development (proposed + construction)'] = km_by_region[['proposed','construction']].sum(axis=1)
km_by_country['in development (proposed + construction)'] = km_by_country[['proposed','construction']].sum(axis=1)

km_by_country = km_by_country[excel_status_list]
km_by_region = km_by_region[excel_status_list]

km_by_region.index.names = ['Region','Subregion']
km_by_country.index.name = 'Country'

km_by_region.loc['Total',:] = km_by_region.sum(axis=0).values
km_by_country.loc['Total',:] = km_by_country.sum(axis=0).values

# drop all-zero rows
km_by_country = km_by_country.loc[~(km_by_country==0).all(axis=1)]

km_by_country.replace(0,'',inplace=True)
km_by_region.replace(0,'',inplace=True)

km_by_region.to_excel(excel_writer, 'Kilometers by region')
km_by_country.to_excel(excel_writer, 'Kilometers by country')

===country-level calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired
===regional calculations===
proposed
construction
shelved
cancelled
operating
idle
mothballed
retired


/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55122/2198785467.py:40: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_region.to_excel(excel_writer, 'Kilometers by region')
/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55122/2198785467.py:41: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_country.to_excel(excel_writer, 'Kilometers by country')


In [43]:
km_by_region

proposed  construction  \
Region Subregion                                 
Asia   Western Asia      3194.71        210.31   
Europe Eastern Europe    4752.45        935.74   
       Northern Europe    972.49          1.20   
       Southern Europe   6765.84        986.00   
       Western Europe     905.55         98.00   
Total                   16591.04       2231.25   

                        in development (proposed + construction)  shelved  \
Region Subregion                                                            
Asia   Western Asia                                      3405.02  1414.69   
Europe Eastern Europe                                    5688.19  1382.55   
       Northern Europe                                    973.69   1000.0   
       Southern Europe                                   7751.84    834.0   
       Western Europe                                    1003.55            
Total                                                   18822.29  4631.24   

                        cancelled  operating     idle  mothballed  retired  
Region Subregion                                                            
Asia   Western Asia       6123.39   13714.23               842.00           
Europe Eastern Europe     3899.23   47760.60  1651.65     3009.82   1778.0  
       Northern Europe    1003.77   25671.94              1822.17   564.27  
       Southern Europe    4436.09   28472.49                17.24           
       Western Europe     2138.20   28615.96     8.35      148.32           
Total                    17600.68  144235.22   1660.0     5839.55  2342.27

## pipeline km by parent company (owner) and project status

### first check that there are no missing projectids

In [44]:
owner_parent_calculations_df = pandas.DataFrame()
# needs country, km in each country columns as well

for idx,row in country_ratios_df_subset.iterrows():
    #print(row.ProjectID)
    parent_string = row.Parent
    #print(parent_string)

    #print(parent_string)
    # if the first letter of the parent_string is a chinese character, and it ends with [100.00%],
    # that means it's a non-researched QCC owner; keep as-is
    if re.findall(r'[\u4e00-\u9fff]+', parent_string[:1]) != [] and parent_string[-9:]=='[100.00%]':
        parent_list = [parent_string.split(' [100.00%]')[0]]
        percent_list = [1.0]
    # otherwise do the rest of the thing
    else:
        parent_list = re.sub(' \[.*?\]', '', parent_string).split('; ') # all entries must have a Owner [%] syntax
        percent_list = [float(i.rstrip('%'))/100. for i in re.findall('\\d+(?:\\.\\d+)?%', parent_string)]

    if parent_list.__len__()!=percent_list.__len__():
        #print(parent_list)
        if percent_list==[]:
            percent_list = [1/parent_list.__len__() for i in parent_list]
        else:
            nmissing = parent_list.__len__()-percent_list.__len__()
            # distribute nans evenly
            total = numpy.nansum(percent_list)
            leftover = 1-total
            percent_list += [leftover/nmissing]*nmissing
    for p_idx,parent in enumerate(parent_list):
        owner_parent_calculations_df = pandas.concat([owner_parent_calculations_df, 
                                                      pandas.DataFrame([{'Parent':parent, 'ProjectID':row.ProjectID, 
                                                                         'FractionOwnership':percent_list[p_idx],
                                                                         # 'ParentHQCountry':parent_metadata_df.loc[parent,'ParentHQCountry'],
                                                                         # 'ParentHQRegion':parent_metadata_df.loc[parent,'ParentHQRegion'],
                                                                         'Country':row.Country,
                                                                         'Status':row.Status,
                                                                         'LengthMergedKmByCountry':row.LengthMergedKmByCountry}])])

owner_parent_calculations_df['KmOwnership'] = owner_parent_calculations_df.FractionOwnership*owner_parent_calculations_df.LengthMergedKmByCountry

In [45]:
owner_parent_calculations_df

,Parent,ProjectID,FractionOwnership,Country,Status,LengthMergedKmByCountry,KmOwnership
0,Eni SpA,P0439,0.5,Tunisia,operating,10.47,5.235
0,unknown,P0439,0.5,Tunisia,operating,10.47,5.235
0,Eni SpA,P0439,0.5,Malta,operating,160.25,80.125
0,unknown,P0439,0.5,Malta,operating,160.25,80.125
0,Eni SpA,P0439,0.5,Libya,operating,282.96,141.480
...,...,...,...,...,...,...,...
0,Gazprom PJSC,P7029,1.0,Belarus,operating,115.00,115.000
0,Gazprom PJSC,P7030,1.0,Belarus,operating,34.00,34.000
0,Gazprom PJSC,P7032,1.0,Belarus,operating,284.90,284.900
0,Snam SpA,P7058,1.0,Italy,construction,44.00,44.000


In [46]:
excel_status_list_with_countries

['Country',
 'proposed',
 'construction',
 'in development (proposed + construction)',
 'shelved',
 'cancelled',
 'operating',
 'idle',
 'mothballed',
 'retired']

In [47]:
#unique_owner_list = owner_parent_calculations_df.Parent.sort_values().unique().tolist()

##################################################
# create km count by owner, status
##################################################
owners_km_by_status_df = \
    owner_parent_calculations_df.groupby(
    ['Parent',
     # 'ParentHQCountry',
     'Status'])[['KmOwnership']].sum().unstack().droplevel(axis=1, level=[0])

owners_km_by_status_df = owners_km_by_status_df.reindex(columns=status_list)
owners_km_by_status_df = owners_km_by_status_df.reset_index().set_index('Parent')
owners_km_by_status_df.columns.name = None

owners_km_by_status_df['in development (proposed + construction)'] = owners_km_by_status_df[['proposed','construction']].sum(axis=1)

owners_km_by_status_df = owners_km_by_status_df.rename(columns={'Parent':'Parent Company',
                                                                          # 'ParentHQCountry':'Country'
                                                               })
# rearrange the order of the columns for output
owners_km_by_status_df = owners_km_by_status_df[excel_status_list]#_with_countries]

# totals_row = owners_ntrains_by_status_df.sum(axis=0, min_count=0)
# totals_row.name = 'Total'
# owners_ntrains_by_status_df = owners_ntrains_by_status_df.append(totals_row)
owners_km_by_status_df.loc['Total',:] = owners_km_by_status_df.sum(axis=0, min_count=0).values
owners_km_by_status_df.loc['Total','Country'] = ''

owners_km_by_status_df = owners_km_by_status_df.replace(numpy.nan, '')
owners_km_by_status_df = owners_km_by_status_df.replace(0, '')

owners_km_by_status_df.to_excel(excel_writer, sheet_name='Kilometers by owner')

### pipeline km by start year, type

In [48]:
pipes_started = pipes_df_touse.copy()
#pipes_started['StartYearLatest'].replace(numpy.nan,'',inplace=True)

if fuel_type == 'Gas':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='Gas')]
if fuel_type == 'Oil':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='Oil')]
if fuel_type == 'NGL':
    pipes_started = pipes_started[(pipes_started['Status'].isin(['operating'])) &
                              (pipes_started['Fuel']=='NGL')]

pipes_started_sum = pipes_started.groupby('StartYearEarliest')['LengthMergedKm'].sum()

In [49]:
pipes_started_sum

StartYearEarliest
1948.0     203.00
1952.0    1301.00
1959.0     114.00
1960.0    1283.80
1961.0     221.28
           ...   
2022.0    2263.00
2023.0     198.00
2024.0       3.20
2026.0     188.00
2030.0     100.00
Name: LengthMergedKm, Length: 68, dtype: float64

In [50]:
if fuel_type == 'Gas':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['Gas pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Gas pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'Oil':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['Oil pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['Oil pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

if fuel_type == 'NGL':
    km_by_start_year = pandas.DataFrame(index=list(range(1980,2025)), columns=['NGL pipeline km'])
    km_by_start_year.index.name = 'Start year'
    km_by_start_year['NGL pipeline km'] = pipes_started_sum
    km_by_start_year.replace(numpy.nan,0,inplace=True)

km_by_start_year.loc['Total',:] = km_by_start_year.sum(axis=0)

km_by_start_year.to_excel(excel_writer, 'Kilometers by start year')
#km_by_start_year

/var/folders/fl/t07mc8053p33mn6mdmvp45580000gn/T/ipykernel_55122/652397138.py:21: FutureWarning: Starting with pandas version 3.0 all arguments of to_excel except for the argument 'excel_writer' will be keyword-only.
  km_by_start_year.to_excel(excel_writer, 'Kilometers by start year')


## save excel file

In [51]:
excel_writer.close()

## calculating stats for landing page

In [52]:
# number of projects tracked in total
print(pipes_df_touse.loc[pipes_df_touse.Fuel==fuel_type].shape[0], 'gas pipeline projects tracked')
print(pipes_df_touse.loc[pipes_df_touse.Fuel==fuel_type]['LengthMergedKm'].sum()/1e6, 'M km tracked')

902 gas pipeline projects tracked
0.21027831 M km tracked


In [53]:
# number of projects tracked in total
print(pipes_df_touse.loc[(pipes_df_touse.Fuel==fuel_type)&
                        (pipes_df_touse.Status.isin(['proposed','construction']))].shape[0], 'gas pipeline projects tracked')
print(pipes_df_touse.loc[(pipes_df_touse.Fuel==fuel_type)&
                        (pipes_df_touse.Status.isin(['proposed','construction']))]['LengthMergedKm'].sum()/1e3, 'K km tracked')

149 gas pipeline projects tracked
24.808049999999998 K km tracked
